# Panoptic Segmentation

## 핵심 논문
- **Panoptic Segmentation** (CVPR 2019, Kirillov): 태스크 정의 + PQ 메트릭 제안
- **Panoptic FPN** (CVPR 2019): FPN backbone + semantic/instance 동시 예측
- **Mask2Former** (CVPR 2022): Transformer query 기반 통합 아키텍처 — 현재 SOTA

## Panoptic vs Semantic vs Instance
| 방식 | 출력 | 차이 |
|------|------|------|
| Semantic Seg | 클래스 레이블 | 같은 클래스는 구분 없음 |
| Instance Seg | 객체 마스크 + 클래스 | stuff(배경) 클래스 처리 불가 |
| **Panoptic Seg** | 클래스 + 인스턴스 ID | **stuff + things 모두 처리** |

- **Things**: 셀 수 있는 객체 (차, 사람) → 개별 ID 필요
- **Stuff**: 셀 수 없는 배경 (도로, 하늘) → 클래스만 필요

## 구현 순서
1. Panoptic 데이터셋 생성 (Things + Stuff)
2. Panoptic FPN 아키텍처 구현 (ResNet + FPN + Semantic Head + Instance Head)
3. Panoptic Quality (PQ) 평가 파이프라인
4. Mask2Former 핵심 아이디어 구현 (Masked Attention + Hungarian)
5. 결과 비교

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.optimize import linear_sum_assignment
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── 클래스 정의 ────────────────────────────────────────────────────────
# Stuff (배경, 셀 수 없음): road, sky, building
# Things (객체, 셀 수 있음): car, pedestrian
STUFF_CLASSES  = {0: 'road', 1: 'sky', 2: 'building'}
THINGS_CLASSES = {3: 'car',  4: 'pedestrian'}
ALL_CLASSES    = {**STUFF_CLASSES, **THINGS_CLASSES}
N_STUFF  = len(STUFF_CLASSES)
N_THINGS = len(THINGS_CLASSES)
N_CLASSES = N_STUFF + N_THINGS  # 5
IMG_H, IMG_W = 256, 256
print(f'클래스 수: {N_CLASSES} (stuff={N_STUFF}, things={N_THINGS})')

## 1. 합성 Panoptic 데이터셋

In [ ]:
class PanopticDataset(Dataset):
    """
    합성 Panoptic 데이터셋.
    GT:
      - semantic_mask (H, W): 클래스 ID (0~4)
      - instance_mask (H, W): instance ID (0=배경, 1~N=things 인스턴스)
      - panoptic_mask (H, W): stuff는 class_id, things는 class_id * 1000 + inst_id
      - boxes (N, 4): things 인스턴스의 GT bbox [x1,y1,x2,y2]
      - labels (N,): things 인스턴스의 클래스 ID
    """
    def __init__(self, n_samples=400, h=IMG_H, w=IMG_W):
        self.n = n_samples
        self.h, self.w = h, w

    def _make_scene(self, seed):
        rng = np.random.default_rng(seed)
        img  = np.zeros((self.h, self.w, 3), dtype=np.float32)
        sem  = np.zeros((self.h, self.w), dtype=np.int64)   # semantic
        inst = np.zeros((self.h, self.w), dtype=np.int64)   # instance

        # ── Stuff: sky (상단 40%) ──
        sky_h = int(self.h * 0.4)
        sky_c = rng.uniform([0.4, 0.6, 0.8], [0.6, 0.8, 1.0])
        img[:sky_h] = sky_c
        sem[:sky_h] = 1  # sky

        # ── Stuff: building (중간 15%) ──
        bld_h = sky_h + int(self.h * 0.15)
        bld_c = rng.uniform([0.4, 0.4, 0.4], [0.7, 0.7, 0.7])
        img[sky_h:bld_h] = bld_c
        sem[sky_h:bld_h] = 2  # building

        # ── Stuff: road (하단) ──
        road_c = rng.uniform([0.3, 0.3, 0.3], [0.5, 0.5, 0.5])
        img[bld_h:] = road_c
        sem[bld_h:] = 0  # road

        # ── Things: cars + pedestrians ──
        boxes, labels = [], []
        inst_id = 1
        n_objs = rng.integers(2, 7)
        for _ in range(n_objs):
            is_car = rng.random() > 0.35
            cls_id = 3 if is_car else 4  # car or pedestrian

            if is_car:
                bh = rng.integers(30, 60)
                bw = rng.integers(50, 90)
            else:
                bh = rng.integers(40, 70)
                bw = rng.integers(15, 30)

            y1 = rng.integers(bld_h, self.h - bh)
            x1 = rng.integers(5, self.w - bw - 5)
            y2, x2 = y1 + bh, x1 + bw

            obj_c = rng.uniform([0.1, 0.1, 0.1], [0.9, 0.9, 0.9])
            img[y1:y2, x1:x2] = obj_c
            sem[y1:y2, x1:x2] = cls_id
            inst[y1:y2, x1:x2] = inst_id
            boxes.append([x1, y1, x2, y2])
            labels.append(cls_id)
            inst_id += 1

        img += np.random.default_rng(seed+1000).normal(0, 0.01, img.shape)
        img = np.clip(img, 0, 1)

        boxes  = np.array(boxes,  dtype=np.float32) if boxes  else np.zeros((0, 4), dtype=np.float32)
        labels = np.array(labels, dtype=np.int64)   if labels else np.zeros((0,),   dtype=np.int64)
        return img, sem, inst, boxes, labels

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        img, sem, inst, boxes, labels = self._make_scene(idx)
        img_t   = torch.from_numpy(img.transpose(2,0,1)).float()
        sem_t   = torch.from_numpy(sem).long()
        inst_t  = torch.from_numpy(inst).long()
        boxes_t = torch.from_numpy(boxes).float()
        labels_t= torch.from_numpy(labels).long()
        return img_t, sem_t, inst_t, boxes_t, labels_t


# 데이터 확인
dataset = PanopticDataset(n_samples=500)
img_t, sem_t, inst_t, boxes_t, labels_t = dataset[0]

COLORS = {
    0: [0.4, 0.4, 0.4],  # road
    1: [0.6, 0.8, 1.0],  # sky
    2: [0.6, 0.6, 0.6],  # building
    3: [1.0, 0.2, 0.2],  # car
    4: [0.2, 1.0, 0.2],  # pedestrian
}
sem_vis = np.zeros((IMG_H, IMG_W, 3))
for cls_id, color in COLORS.items():
    sem_vis[sem_t.numpy() == cls_id] = color

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img_t.permute(1,2,0).numpy())
axes[0].set_title('입력 이미지')
axes[1].imshow(sem_vis)
axes[1].set_title('Semantic GT (5 클래스)')
axes[2].imshow(inst_t.numpy(), cmap='tab20')
axes[2].set_title('Instance GT (ID별 색상)')

# Panoptic 시각화 (인스턴스별로 다른 색)
pan_vis = sem_vis.copy()
np.random.seed(0)
for iid in np.unique(inst_t.numpy()):
    if iid == 0:
        continue
    mask = inst_t.numpy() == iid
    pan_vis[mask] = np.random.rand(3)
axes[3].imshow(pan_vis)
axes[3].set_title('Panoptic GT (stuff=클래스, things=인스턴스별 색)')

legend_patches = [mpatches.Patch(color=c, label=n) for n, c in
                  zip(ALL_CLASSES.values(), COLORS.values())]
axes[1].legend(handles=legend_patches, loc='lower right', fontsize=7)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'데이터셋: {len(dataset)}개, things 인스턴스 수: {len(boxes_t)}')

## 2. Panoptic FPN 구현

**구조:**
```
Image → ResNet backbone → FPN → ┬→ Semantic Head (stuff 예측)
                                └→ Instance Head (things 마스크 + 클래스 예측)
Panoptic Merge: Semantic + Instance → 통합 panoptic 결과
```

In [ ]:
# ── Lightweight ResNet Backbone ──────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch),
        )
    def forward(self, x):
        return F.relu(self.net(x) + x)

class LightBackbone(nn.Module):
    """4개 스케일 feature 출력: C2(1/4), C3(1/8), C4(1/16), C5(1/32)"""
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))  # 1/2
        self.layer1 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            ResBlock(64))   # C2: 1/4
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResBlock(128))  # C3: 1/8
        self.layer3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResBlock(256))  # C4: 1/16
        self.layer4 = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResBlock(256))  # C5: 1/32

    def forward(self, x):
        x  = self.stem(x)
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return c2, c3, c4, c5


# ── FPN ─────────────────────────────────────────────────────────────────
class FPN(nn.Module):
    def __init__(self, in_chs=(64,128,256,256), out_ch=128):
        super().__init__()
        self.lat = nn.ModuleList([nn.Conv2d(c, out_ch, 1) for c in in_chs])
        self.out = nn.ModuleList([nn.Conv2d(out_ch, out_ch, 3, padding=1) for _ in in_chs])

    def forward(self, feats):
        c2, c3, c4, c5 = feats
        p5 = self.lat[3](c5)
        p4 = self.lat[2](c4) + F.interpolate(p5, scale_factor=2, mode='nearest')
        p3 = self.lat[1](c3) + F.interpolate(p4, scale_factor=2, mode='nearest')
        p2 = self.lat[0](c2) + F.interpolate(p3, scale_factor=2, mode='nearest')
        return [self.out[i](p) for i, p in enumerate([p2, p3, p4, p5])]


# ── Semantic Head (stuff) ────────────────────────────────────────────────
class SemanticHead(nn.Module):
    """FPN 4 스케일 → 1/4 해상도 semantic logit (N_CLASSES, H/4, W/4)"""
    def __init__(self, in_ch=128, n_classes=N_CLASSES):
        super().__init__()
        self.convs = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch, 128, 3, padding=1), nn.ReLU(inplace=True)) for _ in range(4)])
        self.cls = nn.Conv2d(128, n_classes, 1)

    def forward(self, fpn_feats):
        # 모든 스케일을 P2 해상도로 upsample 후 합산
        target_size = fpn_feats[0].shape[-2:]
        merged = sum(F.interpolate(self.convs[i](f), size=target_size, mode='bilinear', align_corners=False)
                     for i, f in enumerate(fpn_feats))
        return self.cls(merged)  # (B, N_CLASSES, H/4, W/4)


# ── Instance Head (things) — 단순 Mask Head ──────────────────────────────
class InstanceHead(nn.Module):
    """
    단순화된 instance head:
    P2 feature → 클래스별 이진 마스크 (N_THINGS 채널)
    실제 Mask R-CNN은 RoI + per-instance head이지만,
    여기서는 FCN 스타일로 things 클래스 마스크를 직접 예측
    """
    def __init__(self, in_ch=128, n_things=N_THINGS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64,  3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, n_things, 1)
        )
    def forward(self, p2):
        return self.net(p2)  # (B, N_THINGS, H/4, W/4)


# ── 전체 Panoptic FPN ────────────────────────────────────────────────────
class PanopticFPN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = LightBackbone()
        self.fpn      = FPN()
        self.sem_head = SemanticHead()
        self.ins_head = InstanceHead()

    def forward(self, x):
        feats = self.backbone(x)
        fpn   = self.fpn(feats)
        sem_logit = self.sem_head(fpn)   # (B, N_CLASSES, H/4, W/4)
        ins_logit = self.ins_head(fpn[0]) # (B, N_THINGS, H/4, W/4)
        # 원본 해상도로 upsample
        H, W = x.shape[-2:]
        sem_up = F.interpolate(sem_logit, size=(H,W), mode='bilinear', align_corners=False)
        ins_up = F.interpolate(ins_logit, size=(H,W), mode='bilinear', align_corners=False)
        return sem_up, ins_up

    def panoptic_merge(self, sem_logit, ins_logit, threshold=0.5):
        """
        Panoptic Merging:
        - Semantic prediction → stuff 영역
        - Instance mask (things 클래스) → things 영역 덮어쓰기
        반환: panoptic_mask (H, W) — stuff: class_id, things: class_id*1000 + inst_id
        """
        sem_pred = sem_logit.argmax(dim=1)  # (B, H, W)
        pan_pred = sem_pred.clone()

        ins_prob = torch.sigmoid(ins_logit)  # (B, N_THINGS, H, W)
        for b in range(sem_logit.shape[0]):
            inst_id = 1
            for t_idx in range(N_THINGS):
                mask = ins_prob[b, t_idx] > threshold
                if mask.sum() > 100:  # 최소 크기 필터
                    cls_id = N_STUFF + t_idx  # 3 or 4
                    pan_pred[b][mask] = cls_id * 1000 + inst_id
                    inst_id += 1
        return pan_pred


model = PanopticFPN().to(device)
total = sum(p.numel() for p in model.parameters())
print(f'PanopticFPN 파라미터: {total/1e6:.2f}M')
x_dummy = torch.randn(2, 3, IMG_H, IMG_W).to(device)
sem_out, ins_out = model(x_dummy)
print(f'Semantic head 출력: {sem_out.shape}')   # (2, 5, H, W)
print(f'Instance head 출력: {ins_out.shape}')   # (2, 2, H, W)

## 3. Panoptic Quality (PQ) 메트릭

PQ = SQ × RQ 로 분해됩니다:
- **SQ** (Segmentation Quality): 매칭된 쌍의 평균 IoU
- **RQ** (Recognition Quality): TP / (TP + 0.5×FP + 0.5×FN) — F1 score

In [ ]:
def compute_pq(pred_pan, gt_sem, gt_inst, n_classes=N_CLASSES, n_stuff=N_STUFF, iou_thresh=0.5):
    """
    Panoptic Quality 계산.
    pred_pan: (H,W) — panoptic 예측 (stuff: class_id, things: cls*1000+inst_id)
    gt_sem:   (H,W) — GT semantic
    gt_inst:  (H,W) — GT instance
    반환: pq_per_class, sq_per_class, rq_per_class, pq_mean
    """
    # GT panoptic 구성
    gt_pan = gt_sem.copy()
    things_mask = gt_sem >= n_stuff
    gt_pan[things_mask] = gt_sem[things_mask] * 1000 + gt_inst[things_mask]

    pq_list, sq_list, rq_list = [], [], []

    for cls_id in range(n_classes):
        is_stuff = cls_id < n_stuff

        if is_stuff:
            pred_mask = pred_pan == cls_id
            gt_mask   = gt_pan   == cls_id
            if gt_mask.sum() == 0 and pred_mask.sum() == 0:
                pq_list.append(1.0); sq_list.append(1.0); rq_list.append(1.0)
                continue
            inter = (pred_mask & gt_mask).sum()
            union = (pred_mask | gt_mask).sum()
            iou   = inter / (union + 1e-6)
            pq_list.append(iou); sq_list.append(iou); rq_list.append(1.0 if iou > 0 else 0.0)
        else:
            # Things: instance 단위 매칭
            pred_ids = np.unique(pred_pan)
            pred_ids = pred_ids[(pred_ids // 1000) == cls_id]
            gt_ids   = np.unique(gt_pan)
            gt_ids   = gt_ids[(gt_ids // 1000) == cls_id]

            if len(pred_ids) == 0 and len(gt_ids) == 0:
                pq_list.append(1.0); sq_list.append(1.0); rq_list.append(1.0)
                continue

            # IoU 행렬 계산
            iou_mat = np.zeros((len(gt_ids), len(pred_ids)))
            for gi, g_id in enumerate(gt_ids):
                for pi, p_id in enumerate(pred_ids):
                    gm = gt_pan   == g_id
                    pm = pred_pan == p_id
                    inter = (gm & pm).sum()
                    union = (gm | pm).sum()
                    iou_mat[gi, pi] = inter / (union + 1e-6)

            # Hungarian 매칭
            ri, ci = linear_sum_assignment(-iou_mat)
            tp_iou = [iou_mat[r, c] for r, c in zip(ri, ci) if iou_mat[r, c] >= iou_thresh]
            tp = len(tp_iou)
            fp = len(pred_ids) - tp
            fn = len(gt_ids)   - tp

            sq = np.mean(tp_iou) if tp > 0 else 0.0
            rq = tp / (tp + 0.5 * fp + 0.5 * fn + 1e-6)
            pq_list.append(sq * rq); sq_list.append(sq); rq_list.append(rq)

    pq_per_cls = np.array(pq_list)
    sq_per_cls = np.array(sq_list)
    rq_per_cls = np.array(rq_list)
    return pq_per_cls, sq_per_cls, rq_per_cls, pq_per_cls.mean()


# 초기 (랜덤 가중치) 테스트
img_s, sem_s, inst_s, _, _ = dataset[0]
with torch.no_grad():
    sem_out_s, ins_out_s = model(img_s.unsqueeze(0).to(device))
pan_pred_s = model.panoptic_merge(sem_out_s, ins_out_s)[0].cpu().numpy()
pq, sq, rq, pq_mean = compute_pq(pan_pred_s, sem_s.numpy(), inst_s.numpy())
print(f'초기 PQ (랜덤): {pq_mean:.4f}')

## 4. 학습

In [ ]:
def panoptic_collate(batch):
    """boxes/labels 크기가 다르므로 리스트로 반환."""
    imgs   = torch.stack([b[0] for b in batch])
    sems   = torch.stack([b[1] for b in batch])
    insts  = torch.stack([b[2] for b in batch])
    boxes  = [b[3] for b in batch]
    labels = [b[4] for b in batch]
    return imgs, sems, insts, boxes, labels

n_val = 100
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          collate_fn=panoptic_collate, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,
                          collate_fn=panoptic_collate, num_workers=0)

# Loss: Semantic CE + Instance Binary CE
sem_criterion = nn.CrossEntropyLoss(ignore_index=-1)
ins_criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 30
history = {'train_loss': [], 'val_pq': [], 'val_miou': []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, sems, insts, boxes, labels in train_loader:
        imgs = imgs.to(device)
        sems = sems.to(device)
        insts = insts.to(device)

        sem_logit, ins_logit = model(imgs)

        # Semantic loss
        loss_sem = sem_criterion(sem_logit, sems)

        # Instance loss (things 클래스에 대해 이진 마스크)
        ins_gt = torch.zeros_like(ins_logit)
        for t_idx in range(N_THINGS):
            cls_id = N_STUFF + t_idx
            ins_gt[:, t_idx] = (sems == cls_id).float()
        loss_ins = ins_criterion(ins_logit, ins_gt)

        loss = loss_sem + 0.5 * loss_ins
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    # Validation PQ + mIoU
    model.eval()
    pq_list, miou_list = [], []
    with torch.no_grad():
        for imgs, sems, insts, _, _ in val_loader:
            sem_logit, ins_logit = model(imgs.to(device))
            pan_preds = model.panoptic_merge(sem_logit, ins_logit)
            sem_preds = sem_logit.argmax(dim=1).cpu().numpy()

            for b in range(imgs.shape[0]):
                _, _, _, pq_m = compute_pq(pan_preds[b].cpu().numpy(),
                                           sems[b].numpy(), insts[b].numpy())
                pq_list.append(pq_m)
                # mIoU (semantic)
                ious = []
                for c in range(N_CLASSES):
                    pred_c = sem_preds[b] == c
                    gt_c   = sems[b].numpy() == c
                    inter  = (pred_c & gt_c).sum()
                    union  = (pred_c | gt_c).sum()
                    if union > 0:
                        ious.append(inter / union)
                miou_list.append(np.mean(ious) if ious else 0.0)

    val_pq   = np.mean(pq_list)
    val_miou = np.mean(miou_list)
    history['train_loss'].append(train_loss)
    history['val_pq'].append(val_pq)
    history['val_miou'].append(val_miou)

    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss:.4f} | '
              f'Val PQ: {val_pq:.4f} | mIoU: {val_miou:.4f}')

print('\n학습 완료!')

## 5. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], 'b-')
axes[0].set_title('Train Loss'); axes[0].grid(True)
axes[1].plot(history['val_pq'], 'r-')
axes[1].set_title('Val PQ'); axes[1].set_ylim(0, 1); axes[1].grid(True)
axes[2].plot(history['val_miou'], 'g-')
axes[2].set_title('Val mIoU (semantic)'); axes[2].set_ylim(0, 1); axes[2].grid(True)
plt.tight_layout()
plt.show()

# 정성적 결과
model.eval()
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
samples = [val_ds[i] for i in [0, 5, 10]]
col_titles = ['입력 이미지', 'GT Panoptic', 'Pred Semantic', 'Pred Panoptic']
for ax, ct in zip(axes[0], col_titles):
    ax.set_title(ct, fontsize=10)

with torch.no_grad():
    for row, (img_s, sem_s, inst_s, _, _) in enumerate(samples):
        sem_logit_s, ins_logit_s = model(img_s.unsqueeze(0).to(device))
        pan_pred_s = model.panoptic_merge(sem_logit_s, ins_logit_s)[0].cpu().numpy()
        sem_pred_s = sem_logit_s.argmax(1)[0].cpu().numpy()

        # GT panoptic 시각화
        def make_sem_vis(sem_np):
            vis = np.zeros((IMG_H, IMG_W, 3))
            for c, col in COLORS.items():
                vis[sem_np == c] = col
            return vis

        axes[row, 0].imshow(img_s.permute(1,2,0).numpy())
        axes[row, 1].imshow(make_sem_vis(sem_s.numpy()))
        axes[row, 2].imshow(make_sem_vis(sem_pred_s))
        # panoptic (things별 다른 색)
        pan_vis = make_sem_vis(sem_pred_s).copy()
        np.random.seed(0)
        for uid in np.unique(pan_pred_s):
            if uid > 1000:
                pan_vis[pan_pred_s == uid] = np.random.rand(3)
        axes[row, 3].imshow(pan_vis)
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. 클래스별 PQ 분해 (PQ = SQ × RQ)

In [ ]:
# 전체 val에서 클래스별 평균 PQ/SQ/RQ
pq_all = np.zeros(N_CLASSES)
sq_all = np.zeros(N_CLASSES)
rq_all = np.zeros(N_CLASSES)
count  = 0

model.eval()
with torch.no_grad():
    for imgs, sems, insts, _, _ in val_loader:
        sem_logit, ins_logit = model(imgs.to(device))
        pan_preds = model.panoptic_merge(sem_logit, ins_logit)
        for b in range(imgs.shape[0]):
            pq_c, sq_c, rq_c, _ = compute_pq(pan_preds[b].cpu().numpy(),
                                              sems[b].numpy(), insts[b].numpy())
            pq_all += pq_c; sq_all += sq_c; rq_all += rq_c
            count += 1

pq_all /= count; sq_all /= count; rq_all /= count

print('=' * 65)
print('  클래스별 PQ 분해  (PQ = SQ × RQ)')
print('=' * 65)
print(f'  {"클래스":<12} {"타입":<8} {"PQ":>8} {"SQ":>8} {"RQ":>8}')
print('-' * 65)
for cls_id, cls_name in ALL_CLASSES.items():
    tp = 'stuff' if cls_id < N_STUFF else 'things'
    print(f'  {cls_name:<12} {tp:<8} {pq_all[cls_id]:>8.4f} {sq_all[cls_id]:>8.4f} {rq_all[cls_id]:>8.4f}')
print('-' * 65)
print(f'  {"평균":<12} {"":<8} {pq_all.mean():>8.4f} {sq_all.mean():>8.4f} {rq_all.mean():>8.4f}')
print('=' * 65)

## 7. 최종 결과 요약

In [ ]:
final_pq   = history['val_pq'][-1]
final_miou = history['val_miou'][-1]

print('=' * 65)
print('  skillup_round7 / 02 Panoptic Segmentation — 최종 결과')
print('=' * 65)
print(f'  모델:   PanopticFPN (LightResNet + FPN + Semantic/Instance Head)')
print(f'  데이터: 합성 도시 장면 (train=400, val=100, 5클래스)')
print(f'  PQ:     {final_pq:.4f}  (stuff+things 통합 평가)')
print(f'  mIoU:   {final_miou:.4f}  (semantic baseline)')
print()
print('  클래스별 최종 PQ:')
for cls_id, cls_name in ALL_CLASSES.items():
    tp = '(stuff)' if cls_id < N_STUFF else '(things)'
    print(f'    {cls_name:<12} {tp:<8}: PQ={pq_all[cls_id]:.4f}  SQ={sq_all[cls_id]:.4f}  RQ={rq_all[cls_id]:.4f}')
print()
print('  구현 핵심:')
print('    1. Panoptic = Semantic(stuff) + Instance(things) 통합')
print('    2. PQ = SQ × RQ 분해 — 위치 정확도 vs 탐지 성능 분리')
print('    3. Hungarian 매칭 — things 인스턴스 쌍 최적 할당')
print('    4. Panoptic Merging — 예측 우선순위: things > stuff')
print('    5. Mask2Former와의 차이: query 기반 vs FCN 기반')
print('=' * 65)